In [4]:
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from tensorflow.keras.utils import load_img, img_to_array
import tensorflow as tf

2026-03-01 22:39:24.762438: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
PROJECT_ROOT = Path().resolve().parent

RAW_PATH = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [6]:
image_paths = list(RAW_PATH.glob("*.jpg"))

MAX_IMAGES = 8000  # usamos sólo 8000 imágenes de las 15000 que descargamos (para hacer pruebas)
image_paths = image_paths[:MAX_IMAGES]

print(f"Usando {len(image_paths)} imágenes")

Usando 8000 imágenes


In [7]:
# ahora cargamos las imágenes y redimensionamos
IMG_SIZE = 64
images = []

for path in tqdm(image_paths):
    img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    img = img_to_array(img)
    images.append(img)

images = np.array(images, dtype="float32") / 255.0

print(images.shape)

100%|██████████| 8000/8000 [00:12<00:00, 664.89it/s]


(8000, 64, 64, 3)


In [8]:
# hacemos una versión con blur, que será el input que utilizaremos
# estamos utilizando el blur gaussiando de tensorflow
def apply_blur(images):
    return tf.nn.depthwise_conv2d(
        images,
        tf.ones((5, 5, 3, 1)) / 25.0,
        strides=[1, 1, 1, 1],
        padding="SAME"
    )

images_tf = tf.convert_to_tensor(images)
blurred_images = apply_blur(images_tf)
blurred_images = blurred_images.numpy()

print(blurred_images.shape)

(8000, 64, 64, 3)


In [9]:
# dividimos en train y test, usando las imágenes que blureamos cooo 'X' y las originales como 'y'
X_train, X_test, y_train, y_test = train_test_split(
    blurred_images,
    images,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (6400, 64, 64, 3)
Test: (1600, 64, 64, 3)


In [10]:
# ahora guardamos en la carpeta 'processed'
# utilizamos .npy, en lugar de guardar todas las imágenes, se guarda un archivo binario
np.save(PROCESSED_PATH / "X_train.npy", X_train)
np.save(PROCESSED_PATH / "X_test.npy", X_test)
np.save(PROCESSED_PATH / "y_train.npy", y_train)
np.save(PROCESSED_PATH / "y_test.npy", y_test)